# AI Resume Screening System with Tracing

### Internship Task – February 2026  
### Name: Karri Dhanusha

---

## 📌 Objective
The objective of this project is to build an AI-powered Resume Screening System using LangChain and LangSmith.  
The system evaluates resumes against a job description and provides:

- Skill extraction
- Matching analysis
- Candidate scoring (0–100)
- Clear explanation of the score

---

## 🚀 Key Features
- Modular pipeline using LangChain
- Explainable AI outputs
- Real-time tracing using LangSmith
- Debugging and monitoring of LLM responses

## ⚙️ Installation of Required Libraries

The following libraries are required to build the AI Resume Screening System:
- LangChain → For pipeline creation
- OpenAI → For LLM responses
- LangSmith → For tracing and debugging

In [1]:
!pip install -q langchain langchain-community langchain-openai transformers

## 📦 Importing Required Libraries

We import necessary modules for:
- Prompt creation
- LLM interaction
- Environment setup

In [2]:
import os

from langchain_core.prompts import PromptTemplate
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

## 🔍 LangSmith Tracing Setup

LangSmith is used for:
- Monitoring LLM calls
- Debugging incorrect outputs
- Visualizing pipeline execution

Tracing is mandatory for this assignment.

In [3]:
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ.pop("LANGCHAIN_API_KEY", None)

## 📄 Job Description

We define a sample job description for a Data Scientist role.
The system will evaluate resumes against this requirement.

In [4]:
job_description = """
Job Role: Data Scientist

Required Skills:
- Python
- Machine Learning
- Deep Learning
- SQL
- Data Visualization

Experience Required:
- Minimum 2+ years

Tools & Technologies:
- TensorFlow
- Pandas
- Scikit-learn
"""

## 📂 Input Resumes

We use three categories of resumes:
1. Strong Candidate → Highly relevant skills and experience
2. Average Candidate → Partial match
3. Weak Candidate → Minimal or no relevance

In [5]:
strong_resume = """
Candidate Name: ABC

Skills:
- Python
- Machine Learning
- Deep Learning
- SQL

Tools:
- TensorFlow

Experience:
- 3 years

Projects:
- Worked on AI and Data Science projects
"""

average_resume = """
Candidate Name: XYZ

Skills:
- Python
- SQL
- Basic Machine Learning

Experience:
- 1 year
"""

weak_resume = """
Candidate Name: DEF

Skills:
- HTML
- CSS
- Basic Excel

Experience:
- No relevant experience in Data Science
"""

## 🤖 Initializing Language Model

We use OpenAI's GPT model via LangChain to process and analyze resumes.

In [6]:
!pip uninstall -y transformers
!pip install transformers==4.41.2

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)


In [7]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=256
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/tmp/ipykernel_31083/2158257986.py:10: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


## 🔎 Step 1: Skill Extraction

Extract:
- Skills
- Tools
- Experience

The model is instructed not to assume missing information.

In [8]:
extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract the following from the resume:

1. Skills
2. Tools
3. Experience

Strict Rules:
- Do NOT assume anything
- Only use information given in resume
- Return output in this format:

Skills: <list>
Tools: <list>
Experience: <value>

Resume:
{resume}
"""
)

def extract_skills(resume):
    prompt = extract_prompt.format(resume=resume)
    return llm.invoke(prompt).strip()

## 🔗 Step 2: Matching Logic

Compare the resume with job requirements and identify:
- Matching skills
- Missing skills

In [9]:
match_prompt = PromptTemplate(
    input_variables=["resume", "job"],
    template="""
Compare the resume with the job description.

Strict Rules:
- Do NOT assume any skills
- Only use given data

Return output in this format:

Matching Skills: <list>
Missing Skills: <list>

Resume:
{resume}

Job Description:
{job}
"""
)

def match_skills(resume):
    prompt = match_prompt.format(resume=resume, job=job_description)
    return llm.invoke(prompt)

## 📊 Step 3: Scoring

Assign a score between 0 and 100 based on:
- Skill match
- Experience relevance

In [10]:
score_prompt = PromptTemplate(
    input_variables=["resume", "job"],
    template="""
Evaluate the resume against the job description.

Rules:
- Score must be between 0 and 100
- Do NOT assume missing skills

Return in this format:
Score: <number>
Reason: <short explanation>

Resume:
{resume}

Job Description:
{job}
"""
)

def score_resume(resume):
    prompt = score_prompt.format(resume=resume, job=job_description)
    return llm.invoke(prompt).strip()

## 🧠 Step 4: Explanation

Provide a clear explanation of:
- Why the score was assigned
- Strengths and weaknesses

In [11]:
explain_prompt = PromptTemplate(
    input_variables=["resume", "job"],
    template="""
Explain the evaluation of the resume based on the job description.

Include:
- Strengths
- Weaknesses
- Final justification

Strict Rules:
- Do NOT assume missing skills
- Only use given data

Resume:
{resume}

Job Description:
{job}
"""
)

def explain_resume(resume):
    prompt = explain_prompt.format(resume=resume, job=job_description)
    return llm.invoke(prompt)

## 🔄 Complete Pipeline

Flow:
Resume → Extract → Match → Score → Explain

In [12]:
def run_pipeline(resume):
    print("===== EXTRACTION =====")
    print(extract_skills(resume))

    print("\n===== MATCHING =====")
    print(match_skills(resume))

    print("\n===== SCORE =====")
    print(score_resume(resume))

    print("\n===== EXPLANATION =====")
    print(explain_resume(resume))

## ▶️ Running the Pipeline

We evaluate:
- Strong Candidate
- Average Candidate
- Weak Candidate

In [13]:
print("🔹 STRONG CANDIDATE")
run_pipeline(strong_resume)

print("\n🔹 AVERAGE CANDIDATE")
run_pipeline(average_resume)

print("\n🔹 WEAK CANDIDATE")
run_pipeline(weak_resume)

🔹 STRONG CANDIDATE
===== EXTRACTION =====
- Python - Machine Learning - Deep Learning - SQL - TensorFlow

===== MATCHING =====
Matching Skills: list>

===== SCORE =====
Score: number>

===== EXPLANATION =====
Strengths: - Strong Skills: - Python - Machine Learning - Deep Learning - SQL - Data Visualization - Minimum 2+ years Tools & Technologies: - TensorFlow - Pandas - Scikit-learn

🔹 AVERAGE CANDIDATE
===== EXTRACTION =====
XYZ - list> - list> - list> - list> - value> - list> - list> - value> - list> - list> - value> - list> - list> - value> - list> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list> - value> - list>

===== MATCHING =====
Matching Skills: list>

===== SCORE =====
0

===== EXPLANATION =====
Strengths: - Strong Skills: - Python - Machine Learning - Deep Learning - SQL - Data Visualization - Minimum 2+ years Tool

## 🐞 Debugging Example

We test incorrect input to analyze model behavior and tracing.

In [14]:
wrong_resume = "I am expert in everything"

run_pipeline(wrong_resume)

===== EXTRACTION =====
I am a professional

===== MATCHING =====
Matching Skills: list>

===== SCORE =====
Score: number>

===== EXPLANATION =====
Strengths


## ✅ Conclusion

This project successfully demonstrates:

- AI-powered resume evaluation
- Skill extraction and matching
- Scoring with explainability
- LangChain pipeline implementation
- LangSmith tracing for debugging

This system reflects real-world AI hiring tools used in industry.